# Tree-Based Models: Decision Trees & Random Forest

This notebook implements and evaluates tree-based models to capture nonlinear feature interactions.

**Goal**: Explore the bias-variance tradeoff through tree-based models.

## Models:
1. **Decision Tree**: High variance, low bias - can overfit
2. **Random Forest**: Ensemble of trees - reduces variance through bagging

## Evaluation Metrics:
- **F1-Score** (primary): Mean ± standard deviation across 5 folds
- **ROC-AUC**, **Precision**, **Recall**

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# Import the evaluate_model function
from evaluate_model import evaluate_model

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load preprocessed training data
print("Loading preprocessed training data...")
train_df = pd.read_csv('../data/train_data_preprocessed.csv')
print(f"Training data shape: {train_df.shape}")

# Prepare features and target
X = train_df.drop('Revenue', axis=1)
y = train_df['Revenue']

print(f"\nFeatures shape: {X.shape}")
print(f"Features: {list(X.columns)}")

In [ ]:
# Setup cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 2. Decision Tree

Single decision tree - can capture complex nonlinear patterns but prone to overfitting.

In [ ]:
# Decision Tree with hyperparameter tuning
dt = DecisionTreeClassifier(random_state=42)

# Hyperparameter grid
param_grid_dt = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Grid search
grid_search_dt = GridSearchCV(
    dt, param_grid_dt, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)

print("Training Decision Tree with Grid Search...")
grid_search_dt.fit(X, y)

print(f"\nBest parameters: {grid_search_dt.best_params_}")
print(f"Best F1-Score: {grid_search_dt.best_score_:.4f}")

In [ ]:
# Evaluate best Decision Tree
results_dt = evaluate_model(grid_search_dt.best_estimator_, X, y, cv, "Decision Tree")

## 3. Random Forest

Ensemble of decision trees - reduces variance through bagging and random feature selection.

In [ ]:
# Random Forest with hyperparameter tuning
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Hyperparameter grid
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Grid search
grid_search_rf = GridSearchCV(
    rf, param_grid_rf, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)

print("Training Random Forest with Grid Search...")
print("This may take several minutes...")
grid_search_rf.fit(X, y)

print(f"\nBest parameters: {grid_search_rf.best_params_}")
print(f"Best F1-Score: {grid_search_rf.best_score_:.4f}")

In [ ]:
# Evaluate best Random Forest
results_rf = evaluate_model(grid_search_rf.best_estimator_, X, y, cv, "Random Forest")

## 4. Feature Importance Analysis

In [ ]:
# Get feature importances from Random Forest
import matplotlib.pyplot as plt

feature_importances = pd.DataFrame({
    'feature': X.columns,
    'importance': grid_search_rf.best_estimator_.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importances.head(10))

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(feature_importances.head(10)['feature'], feature_importances.head(10)['importance'])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Model Comparison

In [ ]:
print(f"\n{'='*60}")
print("TREE-BASED MODEL COMPARISON")
print(f"{'='*60}")
print(f"\n{'Metric':<15} {'Decision Tree':<15} {'Random Forest':<15}")
print(f"{'-'*45}")
print(f"{'F1-Score':<15} {results_dt['cv_f1']:<15.4f} {results_rf['cv_f1']:<15.4f}")
print(f"{'Precision':<15} {results_dt['cv_precision']:<15.4f} {results_rf['cv_precision']:<15.4f}")
print(f"{'Recall':<15} {results_dt['cv_recall']:<15.4f} {results_rf['cv_recall']:<15.4f}")
print(f"{'ROC-AUC':<15} {results_dt['cv_roc_auc']:<15.4f} {results_rf['cv_roc_auc']:<15.4f}")

print(f"\n{'='*60}")
print("KEY INSIGHTS")
print(f"{'='*60}")
print(f"\n1. Decision Tree:")
print(f"   - Best max_depth: {grid_search_dt.best_params_['max_depth']}")
print(f"   - Can capture complex patterns but may overfit")
print(f"\n2. Random Forest:")
print(f"   - Best n_estimators: {grid_search_rf.best_params_['n_estimators']}")
print(f"   - Best max_depth: {grid_search_rf.best_params_['max_depth']}")
print(f"   - Reduces variance through ensemble averaging")
print(f"   - Captures nonlinear interactions automatically")

## 6. Summary

**Bias-Variance Tradeoff:**
- **Decision Tree**: Low bias, high variance (prone to overfitting)
- **Random Forest**: Low bias, reduced variance (through bagging)

**Key Findings:**
- Tree-based models naturally capture nonlinear feature interactions
- Random Forest typically outperforms single Decision Tree
- Feature importance reveals which variables drive predictions

**Next Steps:**
- Compare with gradient boosting (XGBoost)
- Explore neural networks for flexible nonlinear modeling